In [1]:
# DIVERGES FROM SUBSTRATE cell 0 — sys.path adjusted for notebooks/ location (one parent dir up vs replication/notebooks/).
import os, sys
_HERE = os.path.abspath(os.getcwd())
for p in (os.path.abspath(os.path.join(_HERE, "..", "reference")),
          os.path.abspath(os.path.join(_HERE, "..", "replication"))):
    if p not in sys.path:
        sys.path.insert(0, p)

# networkx 3.x removed from_numpy_matrix; reference/stats_count.py still calls it.
# Behavior of from_numpy_array on a 2D ndarray matches the old from_numpy_matrix.
import networkx as _nx
if not hasattr(_nx, "from_numpy_matrix"):
    _nx.from_numpy_matrix = _nx.from_numpy_array


<!-- DIVERGES FROM SUBSTRATE cell 1 — repointed paths, per-run disk math updated to include embedding cost (16v.2: ~110 MB per (lang, color)), manual-rerun note. -->
## Disk budget

This notebook persists attention tensors to `../data/phase3/attentions/` before
extracting topology features. Features are saved to `../results/phase3_thresholds/`.
Embeddings are saved to `../data/phase3/embeddings/`.

Per-run estimate with `max_tokens=32` (KWIC contexts ~21 whitespace tokens):
- **Attention** is `batch × 12 layers × 12 heads × 32 × 32 × 2 bytes` ≈ 295 KB per sample.
  en/color has ~2200 samples → **~650 MB per (lang, color)** (deleted at end by default — `KEEP_ATTENTION=False`).
- **Embeddings (16v)** are `batch × 32 × 768 × 2 bytes` ≈ 49 KB per sample.
  en/color → **~110 MB per (lang, color)** (retained by default — `KEEP_EMBEDDINGS=True`, since downstream Draganov-style sidecar will consume them).
- **Three runs (en/ru/es)** without attention cleanup ≈ 2 GB; embeddings retained ≈ 330 MB. Plan accordingly.

The final cell deletes the per-(lang, domain) attention files after features are
saved. Set `KEEP_ATTENTION = True` in that cell to retain them — useful
when chaining into `features_calculation_ripser_and_templates.ipynb` on
the same (lang, domain) to avoid recomputing attention.
Set `KEEP_EMBEDDINGS = True` (the default) to retain embedding artifacts for
downstream Draganov-style analysis.

### ⚠️ STALE-KERNEL HAZARD — read before re-running

To process all (lang, domain) pairs, edit `lang` in cell 10 and re-run the
notebook. This mirrors how the substrate is run for train/valid/test — one
pass per pair.

**Use *Run All Below* from cell 10 (or restart the kernel) — do NOT use
*Run Selected Cell* on cell 10 alone.** If you re-run cell 10 only, then
trigger cell 26 (the attention extraction loop), the notebook will write
attention matrices computed from the *previous* language's `data` /
`batched_sentences` (still in kernel memory) under the *new* language's
filename. The shape and counts will look right; the contents will be wrong.
Restart the kernel between languages if you want a clean room.


In [2]:
# DIVERGES FROM SUBSTRATE cell 2 — add BertTokenizerFast (needed for word_ids alignment); switch to grab_attention_and_embeddings (16v.1) for joint attention+embedding extraction.
from collections import defaultdict
import itertools
import re
import subprocess

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from transformers import BertTokenizer, BertTokenizerFast, BertModel, BertForSequenceClassification

from stats_count import *
from grab_weights_compat import grab_attention_and_embeddings, text_preprocessing


In [3]:
import warnings

warnings.filterwarnings('ignore')

In [4]:
!env | grep CUDA_VISIBLE

## Parameters

In [5]:
np.random.seed(42) # For reproducibility.

In [6]:
# DIVERGES FROM SUBSTRATE cell 7 — max_tokens 128→32 for KWIC contexts; model bert-base-uncased→bert-base-multilingual-cased.
max_tokens_amount  = 32  # KWIC contexts are ~21 whitespace tokens; 32 leaves headroom
                         # for [CLS]/[SEP] and ru/es subword expansion.
stats_cap          = 500 # Max value that the feature can take. Is NOT applicable to Betty numbers.
    
layers_of_interest = [i for i in range(12)]  # Layers for which attention matrices and features on them are 
                                             # calculated. For calculating features on all layers, leave it be
                                             # [i for i in range(12)].
stats_name = "s_e_v_c_b0b1" # The set of topological features that will be count (see explanation below)

thresholds_array = [0.025, 0.05, 0.1, 0.25, 0.5, 0.75] # The set of thresholds
thrs = len(thresholds_array)                           # ("t" in the paper)

model_path = tokenizer_path = "bert-base-multilingual-cased"

# You can use either standard or fine-tuned BERT. If you want to use fine-tuned BERT to your current task, save the
# model and the tokenizer with the commands tokenizer.save_pretrained(output_dir); 
# bert_classifier.save_pretrained(output_dir) into the same directory and insert the path to it here.


### Explanation of stats_name parameter

Currently, we implemented calculation of the following graphs features:
* "s"    - amount of strongly connected components
* "w"    - amount of weakly connected components
* "e"    - amount of edges
* "v"    - average vertex degree
* "c"    - amount of (directed) simple cycles
* "b0b1" - Betti numbers

The variable stats_name contains a string with the names of the features, which you want to calculate. The format of the string is the following:

"stat_name + "_" + stat_name + "_" + stat_name + ..."

**For example**:

`stats_name == "s_w"` means that the number of strongly and weakly connected components will be calculated

`stats_name == "b0b1"` means that only the Betti numbers will be calculated

`stats_name == "b0b1_c"` means that Betti numbers and the number of simple cycles will be calculated

e.t.c.

## Filenames

In [7]:
# DIVERGES FROM SUBSTRATE cell 10 — subset variable replaced with hardcoded lang/domain; paths repointed to data/kwic/, data/phase3/attentions/, results/phase3_thresholds/; emb_dir / e_file / manifest_file added (16v.2).
# ---------------------------------------------------------------------------
# SCOPE — COSI 115a final analysis is COLOR-only (rescoped 2026-05-04).
# Emotion and kinship CSVs exist under data/kwic/ but are out of scope.
# See `bd show ph-project-mwk` and CLAUDE.md "Current scope" for rationale.
#
# To process all (lang, domain) pairs, edit `lang` below and re-run the
# notebook. This mirrors how the substrate is run for train/valid/test —
# one pass per pair, kernel restart between passes recommended.
# ---------------------------------------------------------------------------
lang   = "ru"        # one of: "en", "ru", "es"
domain = "color"     # COSI 115a scope: color only

input_dir  = "../data/kwic/"
attn_dir   = "../data/phase3/attentions/"
emb_dir    = "../data/phase3/embeddings/"
feat_dir   = "../results/phase3_thresholds/"
os.makedirs(attn_dir, exist_ok=True)
os.makedirs(emb_dir, exist_ok=True)
os.makedirs(feat_dir, exist_ok=True)

prefix = feat_dir + lang + "_" + domain

r_file     = attn_dir + lang + "_" + domain + "_all_heads_" + str(len(layers_of_interest)) + "_layers_MAX_LEN_" + \
             str(max_tokens_amount) + "_" + model_path.split("/")[-1]
# Name of the file for attention matrices weights

e_file     = emb_dir + lang + "_" + domain + "_final_layer_MAX_LEN_" + \
             str(max_tokens_amount) + "_" + model_path.split("/")[-1]
# Name of the file for final-layer embedding matrices

manifest_file = emb_dir + lang + "_" + domain + "_manifest.parquet"
# Per-row manifest with target wordpiece spans

stats_file = feat_dir + lang + "_" + domain + "_all_heads_" + str(len(layers_of_interest)) + "_layers_" + stats_name \
             + "_lists_array_" + str(thrs) + "_thrs_MAX_LEN_" + str(max_tokens_amount) + \
             "_" + model_path.split("/")[-1] + '.npy'
# Name of the file for topological features array


In [8]:
stats_file

'../results/phase3_thresholds/ru_color_all_heads_12_layers_s_e_v_c_b0b1_lists_array_6_thrs_MAX_LEN_32_bert-base-multilingual-cased.npy'

In [9]:
r_file

'../data/phase3/attentions/ru_color_all_heads_12_layers_MAX_LEN_32_bert-base-multilingual-cased'

.csv file must contain the column with the name **sentence** with the texts. It can also contain the column **labels**, which will be needed for testing. Any other arbitrary columns will be ignored.

In [10]:
# DIVERGES FROM SUBSTRATE cell 14 — input path subset+'.csv' → lang+'/'+domain+'.csv'.
try:
    data = pd.read_csv(input_dir + lang + "/" + domain + ".csv").reset_index(drop=True)
except:
    #data = pd.read_csv(input_dir + lang + "/" + domain + ".tsv", delimiter="\t")
    data = pd.read_csv(input_dir + lang + "/" + domain + ".tsv", delimiter="\t", header=None)
    data.columns = ["0", "labels", "2", "sentence"]


In [11]:
data.head()

,term,labels,sentence,target_idx,corpus_source
0,чёрный,чёрный,"Взял черный пиджак и так… Но я вам скажу, что ...",1,rus_news_2020_1M
1,чёрный,чёрный,"Один из фигурантов черного списка, полпред пре...",3,rus_news_2020_1M
2,чёрный,чёрный,Египта Абдель Фаттах ас-Сиси призвал Россию во...,10,rus_news_2023_1M
3,чёрный,чёрный,Перед фотографами Гамильтон позировала в черно...,5,rus_news_2019_1M
4,чёрный,чёрный,Мужчина и женщина на черной иномарке пытались ...,4,rus_news_2019_1M


In [12]:
sentences = data['sentence']
print("Average amount of words in example:", \
      np.mean(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))
print("Max. amount of words in example:", \
      np.max(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))
print("Min. amount of words in example:", \
      np.min(list(map(len, map(lambda x: re.sub('\w', ' ', x).split(" "), data['sentence'])))))

Average amount of words in example: 103.52624614027349
Max. amount of words in example: 182
Min. amount of words in example: 32


In [13]:
def get_token_length(batch_texts):
    inputs = tokenizer(list(batch_texts),
       return_tensors='pt',
       add_special_tokens=True,
       max_length=MAX_LEN,             # Max length to truncate/pad
       padding="max_length",         # Pad sentence to max length
       truncation=True
    )
    inputs = inputs['input_ids'].numpy()
    n_tokens = []
    indexes = np.argwhere(inputs == tokenizer.pad_token_id)
    for i in range(inputs.shape[0]):
        ids = indexes[(indexes == i)[:, 0]]
        if not len(ids):
            n_tokens.append(MAX_LEN)
        else:
            n_tokens.append(ids[0, 1])
    return n_tokens

In [14]:
# DIVERGES FROM SUBSTRATE cell 18 — do_lower_case True→False (Russian/Spanish casing carries meaning); BertTokenizer → BertTokenizerFast (needed for word_ids alignment, 16v).
MAX_LEN = max_tokens_amount
tokenizer = BertTokenizerFast.from_pretrained(tokenizer_path, do_lower_case=False)  # Russian and Spanish casing carries meaning.


In [15]:
data['tokenizer_length'] = get_token_length(data['sentence'].values)

In [16]:
data

,term,labels,sentence,target_idx,corpus_source,tokenizer_length
0,чёрный,чёрный,"Взял черный пиджак и так… Но я вам скажу, что ...",1,rus_news_2020_1M,26
1,чёрный,чёрный,"Один из фигурантов черного списка, полпред пре...",3,rus_news_2020_1M,28
2,чёрный,чёрный,Египта Абдель Фаттах ас-Сиси призвал Россию во...,10,rus_news_2023_1M,32
3,чёрный,чёрный,Перед фотографами Гамильтон позировала в черно...,5,rus_news_2019_1M,32
4,чёрный,чёрный,Мужчина и женщина на черной иномарке пытались ...,4,rus_news_2019_1M,30
...,...,...,...,...,...,...
2262,серый,серый,"полиции, когда мужчина уходил из дома, он был ...",10,rus_news_2020_1M,32
2263,серый,серый,"увидеть и показать всей стране, как живут сего...",10,rus_news_2020_1M,32
2264,серый,серый,"Однако, используя специально разработанные изо...",8,rus_news_2020_1M,32
2265,серый,серый,Был одет в желтую футболку и темно-синие шорты...,10,rus_news_2019_1M,32


In [17]:
ntokens_array = data['tokenizer_length'].values

## Attention extraction

Loading **BERT** and tokenizers using **transformers** library.

In [18]:
from math import ceil

batch_size = 10 # batch size
number_of_batches = ceil(len(data['sentence']) / batch_size)
DUMP_SIZE = 100 # number of batches to be dumped
batched_sentences = np.array_split(data['sentence'].values, number_of_batches)
number_of_files = ceil(number_of_batches / DUMP_SIZE)
adj_matricies = []
adj_filenames = []
assert number_of_batches == len(batched_sentences) # sanity check

In [19]:
# DIVERGES FROM SUBSTRATE cell 25 — BertForSequenceClassification→BertModel (no classifier head needed); do_lower_case True→False; transformers logger silenced around load; + output_hidden_states=True for joint attention+embedding extraction (16v); + slow → fast tokenizer for word_ids alignment.
import transformers as _transformers
device='cuda:0'
_transformers.logging.set_verbosity_error()  # Suppress expected pretraining-head warnings when loading mBERT as BertModel.
model = BertModel.from_pretrained(model_path, output_attentions=True, output_hidden_states=True)
_transformers.logging.set_verbosity_warning()
tokenizer = BertTokenizerFast.from_pretrained(tokenizer_path, do_lower_case=False)  # Russian and Spanish casing carries meaning; fast tokenizer needed for word_ids alignment.
model = model.to(device)
MAX_LEN = max_tokens_amount


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [20]:
# DIVERGES FROM SUBSTRATE cell 26 — joint attention + embedding extraction (16v); manifest rows recorded inline; per-part offset counter for manifest.
from math import ceil

adj_matricies = []
adj_filenames = []
emb_matrices  = []
emb_filenames = []
manifest_rows = []

batch_idx_offset    = 0  # cumulative kwic_row_id at start of next batch
part_start_kwic_id  = 0  # kwic_row_id at which the CURRENT part begins
current_part_number = 1

for i in tqdm(range(number_of_batches), desc="Weights+emb calc"):
    batch_sents = batched_sentences[i]
    attn, emb, wids = grab_attention_and_embeddings(
        model, tokenizer, batch_sents, max_tokens_amount, device
    )
    # attn shape: (n_layers, batch, n_heads, MAX_LEN, MAX_LEN)
    # emb shape:  (batch, MAX_LEN, hidden_size)
    adj_matricies.append(attn)
    emb_matrices.append(emb)

    # Manifest rows for this batch
    for j in range(len(batch_sents)):
        kwic_row_id = batch_idx_offset + j
        target_idx_ws = int(data['target_idx'].iloc[kwic_row_id])
        wp_positions = [
            k for k, wid in enumerate(wids[j])
            if wid == target_idx_ws
        ]
        target_wp_start = wp_positions[0] if wp_positions else -1
        target_wp_end   = wp_positions[-1] if wp_positions else -1
        manifest_rows.append({
            "kwic_row_id":      kwic_row_id,
            "lang":             lang,
            "domain":           domain,
            "term":             data['term'].iloc[kwic_row_id],
            "target_idx":       target_idx_ws,
            "target_wp_start":  target_wp_start,
            "target_wp_end":    target_wp_end,
            "embedding_part":   current_part_number,
            "embedding_offset": kwic_row_id - part_start_kwic_id,
        })
    batch_idx_offset += len(batch_sents)

    if (i+1) % DUMP_SIZE == 0:
        # --- attention dump (existing logic, unchanged for filename) ---
        print(f'Saving: shape {adj_matricies[0].shape}')
        adj_arr = np.concatenate(adj_matricies, axis=1)
        print("Concatenated")
        adj_arr = np.swapaxes(adj_arr, axis1=0, axis2=1)  # sample X layer X head X n_token X n_token
        a_filename = r_file + "_part" + str(ceil(i/DUMP_SIZE)) + "of" + str(number_of_files) + '.npy'
        print(f"Saving weights to: {a_filename}")
        adj_filenames.append(a_filename)
        np.save(a_filename, adj_arr)
        adj_matricies = []
        # --- embedding dump (NEW) — same filename formula so part numbers stay aligned ---
        emb_arr = np.concatenate(emb_matrices, axis=0)
        e_filename = e_file + "_part" + str(ceil(i/DUMP_SIZE)) + "of" + str(number_of_files) + '.npy'
        print(f"Saving embeddings to: {e_filename}")
        np.save(e_filename, emb_arr)
        emb_filenames.append(e_filename)
        emb_matrices = []
        # advance part bookkeeping
        current_part_number += 1
        part_start_kwic_id = batch_idx_offset

# Final partial dump (mirror existing leftover branch — same filename formula as substrate)
if adj_matricies:
    print(f'Saving: shape {adj_matricies[0].shape}')
    adj_arr = np.concatenate(adj_matricies, axis=1)
    print("Concatenated")
    adj_arr = np.swapaxes(adj_arr, axis1=0, axis2=1)  # sample X layer X head X n_token X n_token
    a_filename = r_file + "_part" + str(ceil(i/DUMP_SIZE)) + "of" + str(number_of_files) + '.npy'
    print(f"Saving weights to: {a_filename}")
    adj_filenames.append(a_filename)
    np.save(a_filename, adj_arr)
    emb_arr = np.concatenate(emb_matrices, axis=0)
    e_filename = e_file + "_part" + str(ceil(i/DUMP_SIZE)) + "of" + str(number_of_files) + '.npy'
    print(f"Saving embeddings to: {e_filename}")
    np.save(e_filename, emb_arr)
    emb_filenames.append(e_filename)

# Save manifest (once, after loop completes)
import pandas as _pd
_pd.DataFrame(manifest_rows).to_parquet(manifest_file, index=False)
print(f"Manifest saved: {manifest_file} ({len(manifest_rows)} rows)")

print("Results saved.")


Weights+emb calc:   0%|          | 0/227 [00:00<?, ?it/s]

Saving: shape (12, 10, 12, 32, 32)
Concatenated
Saving weights to: ../data/phase3/attentions/ru_color_all_heads_12_layers_MAX_LEN_32_bert-base-multilingual-cased_part1of3.npy
Saving embeddings to: ../data/phase3/embeddings/ru_color_final_layer_MAX_LEN_32_bert-base-multilingual-cased_part1of3.npy
Saving: shape (12, 10, 12, 32, 32)
Concatenated
Saving weights to: ../data/phase3/attentions/ru_color_all_heads_12_layers_MAX_LEN_32_bert-base-multilingual-cased_part2of3.npy
Saving embeddings to: ../data/phase3/embeddings/ru_color_final_layer_MAX_LEN_32_bert-base-multilingual-cased_part2of3.npy
Saving: shape (12, 10, 12, 32, 32)
Concatenated
Saving weights to: ../data/phase3/attentions/ru_color_all_heads_12_layers_MAX_LEN_32_bert-base-multilingual-cased_part3of3.npy
Saving embeddings to: ../data/phase3/embeddings/ru_color_final_layer_MAX_LEN_32_bert-base-multilingual-cased_part3of3.npy
Manifest saved: ../data/phase3/embeddings/ru_color_manifest.parquet (2267 rows)
Results saved.


## Calculating topological features

In [21]:
stats_name.split("_")

['s', 'e', 'v', 'c', 'b0b1']

In [22]:
# DIVERGES FROM SUBSTRATE cell 29 — output_dir + 'attentions/' → attn_dir (variable renamed in cell 10); logic unchanged.
import os
from multiprocessing import Pool
from tqdm import tqdm

adj_filenames = [
    attn_dir + filename 
    for filename in os.listdir(attn_dir) if r_file in (attn_dir + filename)
]
# sorted by part number
adj_filenames = sorted(adj_filenames, key = lambda x: int(x.split('_')[-1].split('of')[0][4:].strip())) 
adj_filenames


['../data/phase3/attentions/ru_color_all_heads_12_layers_MAX_LEN_32_bert-base-multilingual-cased_part1of3.npy',
 '../data/phase3/attentions/ru_color_all_heads_12_layers_MAX_LEN_32_bert-base-multilingual-cased_part2of3.npy',
 '../data/phase3/attentions/ru_color_all_heads_12_layers_MAX_LEN_32_bert-base-multilingual-cased_part3of3.npy']

In [23]:
# What is calculated in "f(v)". You can add any other function from the array with vertex degrees.

def function_for_v(list_of_v_degrees_of_graph):
    return sum(map(lambda x: np.sqrt(x*x), list_of_v_degrees_of_graph))

def split_matricies_and_lengths(adj_matricies, ntokens_array, num_of_workers):
    splitted_adj_matricies = np.array_split(adj_matricies, num_of_workers)
    splitted_ntokens = np.array_split(ntokens_array, num_of_workers)
    assert all([len(m)==len(n) for m, n in zip(splitted_adj_matricies, splitted_ntokens)]), "Split is not valid!"
    return zip(splitted_adj_matricies, splitted_ntokens)

In [24]:
num_of_workers = 20
pool = Pool(num_of_workers)

In [25]:
stats_tuple_lists_array = []
for i, filename in enumerate(tqdm(adj_filenames, desc='Вычисление признаков')):
    adj_matricies = np.load(filename, allow_pickle=True)
    ntokens = ntokens_array[i*batch_size*DUMP_SIZE : (i+1)*batch_size*DUMP_SIZE]
    splitted = split_matricies_and_lengths(adj_matricies, ntokens, num_of_workers)
    args = [(m, thresholds_array, ntokens, stats_name.split("_"), stats_cap) for m, ntokens in splitted]
    stats_tuple_lists_array_part = pool.starmap(
        count_top_stats, args
    )
    stats_tuple_lists_array.append(np.concatenate([_ for _ in stats_tuple_lists_array_part], axis=3))

Вычисление признаков: 100%|██████████| 3/3 [03:00<00:00, 60.30s/it]


In [26]:
stats_tuple_lists_array = np.concatenate(stats_tuple_lists_array, axis=3)

In [27]:
stats_tuple_lists_array.shape

(12, 12, 6, 2267, 6)

In [28]:
from numpy import inf

np.sum(stats_tuple_lists_array[stats_tuple_lists_array == -inf]) + \
np.sum(stats_tuple_lists_array[stats_tuple_lists_array == inf])

0.0

In [29]:
stats_file

'../results/phase3_thresholds/ru_color_all_heads_12_layers_s_e_v_c_b0b1_lists_array_6_thrs_MAX_LEN_32_bert-base-multilingual-cased.npy'

In [30]:
np.save(stats_file, stats_tuple_lists_array)

##### Checking the size of features matrices:

Layers amount **Х** Heads amount **Х** Features amount **X** Examples amount **Х** Thresholds amount

**For example**:

`stats_name == "s_w"` => `Features amount == 2`

`stats_name == "b0b1"` => `Features amount == 2`

`stats_name == "b0b1_c"` => `Features amount == 3`

e.t.c.

`thresholds_array == [0.025, 0.05, 0.1, 0.25, 0.5, 0.75]` => `Thresholds amount == 6`

In [31]:
stats_tuple_lists_array.shape

(12, 12, 6, 2267, 6)

In [32]:
# DIVERGES FROM SUBSTRATE cell 41 — log message uses (lang, domain) instead of subset; cleanup logic extended to also gate embedding artifacts (16v).
# Delete per-(lang, domain) attention .npy files now that features are saved.
# Set KEEP_ATTENTION = True to retain them (e.g. to skip recompute when
# running features_calculation_ripser_and_templates.ipynb on the same
# (lang, domain)). See the disk-budget cell at the top.
# Embeddings are the whole point of the sidecar emit, so default is KEEP_EMBEDDINGS=True (unlike attentions).
KEEP_ATTENTION  = True
KEEP_EMBEDDINGS = True

import os as _os
if KEEP_ATTENTION:
    print(f"Cleanup: KEEP_ATTENTION=True, retaining {len(adj_filenames)} attention file(s) for ({lang!r}, {domain!r}).")
else:
    _removed = 0
    for _f in adj_filenames:
        if _os.path.exists(_f):
            _os.remove(_f)
            _removed += 1
    print(f"Cleanup: removed {_removed} attention file(s) for ({lang!r}, {domain!r}).")

if KEEP_EMBEDDINGS:
    print(f"Cleanup: KEEP_EMBEDDINGS=True, retaining {len(emb_filenames)} embedding file(s) + manifest for ({lang!r}, {domain!r}).")
else:
    _removed = 0
    for _f in emb_filenames + [manifest_file]:
        if _os.path.exists(_f):
            _os.remove(_f)
            _removed += 1
    print(f"Cleanup: removed {_removed} embedding artifact(s) for ({lang!r}, {domain!r}).")


Cleanup: KEEP_ATTENTION=True, retaining 3 attention file(s) for ('ru', 'color').
Cleanup: KEEP_EMBEDDINGS=True, retaining 3 embedding file(s) + manifest for ('ru', 'color').
